<a href="https://colab.research.google.com/github/andrewenvironmental/ace26-genai/blob/furtman-patch-1/Simple_LLM_Function_Calling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!!pip install litellm

# Important!!!
#
# <---- Set your 'OPENAI_API_KEY' as a secret over there with the "key" icon
#
#
import os
from google.colab import userdata
api_key = userdata.get('ANTHROPIC_API_KEY')
os.environ['ANTHROPIC_API_KEY'] = api_key

In [8]:
import json
import os
from typing import List

from litellm import completion

# def list_files() -> List[str]:
#     """List files in the current directory."""
#     return os.listdir(".")

# def read_file(file_name: str) -> str:
#     """Read a file's contents."""
#     try:
#         with open(file_name, "r") as file:
#             return file.read()
#     except FileNotFoundError:
#         return f"Error: {file_name} not found."
#     except Exception as e:
#         return f"Error: {str(e)}"


# tool_functions = {
#     "list_files": list_files,
#     "read_file": read_file
# }

# tools = [
#     {
#         "type": "function",
#         "function": {
#             "name": "list_files",
#             "description": "Returns a list of files in the directory.",
#             "parameters": {"type": "object", "properties": {}, "required": []}
#         }
#     },
#     {
#         "type": "function",
#         "function": {
#             "name": "read_file",
#             "description": "Reads the content of a specified file in the directory.",
#             "parameters": {
#                 "type": "object",
#                 "properties": {"file_name": {"type": "string"}},
#                 "required": ["file_name"]
#             }
#         }
#     }
# ]

# Our rules are simplified since we don't have to worry about getting a specific output format
agent_rules = [{
    "role": "system",
    "content": """
You are an AI agent that can perform tasks by using available tools.

"""
}]

# user_task = input("What would you like me to do? ")

# memory = [{"role": "user", "content": user_task}]

# messages = agent_rules + memory

# response = completion(
#     model="claude-sonnet-4-5-20250929",
#     messages=messages,
#     tools=tools,
#     max_tokens=1024
# )

# # Extract the tool call from the response, note we don't have to parse now!
# tool = response.choices[0].message.tool_calls[0]
# tool_name = tool.function.name
# tool_args = json.loads(tool.function.arguments)
# result = tool_functions[tool_name](**tool_args)

# print(f"Tool Name: {tool_name}")
# print(f"Tool Arguments: {tool_args}")
# print(f"Result: {result}")

In [13]:
import os
import random
from typing import List, Dict, Any

# --- Tool Implementations ---

def create_secret_message(filename: str, message: str) -> str:
    """Creates a text file containing a secret spy message."""
    try:
        if not filename.endswith(".txt"):
            filename += ".txt"
        with open(filename, "w") as file:
            file.write(f"CLASSIFIED MESSAGE:\n{message}")
        return f"Success! Secret message safely locked away in '{filename}'."
    except Exception as e:
        return f"Mission failed: {str(e)}"

def read_secret_message(filename: str) -> str:
    """Reads and decrypts the secret message from a file."""
    try:
        if not filename.endswith(".txt"):
            filename += ".txt"
        with open(filename, "r") as file:
            return file.read()
    except FileNotFoundError:
        return f"Error: The file '{filename}' vanished or never existed!"
    except Exception as e:
        return f"Error reading file: {str(e)}"

def generate_lucky_numbers(count: int = 5) -> List[int]:
    """Generates a list of random lucky numbers between 1 and 100."""
    count = max(1, min(count, 20))
    return [random.randint(1, 100) for _ in range(count)]

def list_vault_files() -> List[str]:
    """Lists all the secret text files in the current vault directory."""
    return [f for f in os.listdir(".") if f.endswith(".txt")]

def terminate() -> str:
    """Terminates the current active session and signs off."""
    # You can catch this specific string in your main application loop to run `break` or `sys.exit()`
    return "SESSION_TERMINATED"


# --- Tool Mapping ---

tool_functions = {
    "create_secret_message": create_secret_message,
    "read_secret_message": read_secret_message,
    "generate_lucky_numbers": generate_lucky_numbers,
    "list_vault_files": list_vault_files,
    "terminate": terminate
}


# --- OpenAI / LLM Tool Definitions ---

tools = [
    {
        "type": "function",
        "function": {
            "name": "create_secret_message",
            "description": "Creates a new text (.txt) file with a secret message or note.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "The name of the file (e.g., 'blueprint.txt')"},
                    "message": {"type": "string", "description": "The secret text payload to write inside the file."}
                },
                "required": ["filename", "message"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "read_secret_message",
            "description": "Reads and decodes the contents of a specific secret text file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "The name of the text file to read."}
                },
                "required": ["filename"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "generate_lucky_numbers",
            "description": "Generates a list of random lucky numbers for the user.",
            "parameters": {
                "type": "object",
                "properties": {
                    "count": {"type": "integer", "description": "How many lucky numbers to generate. Defaults to 5."}
                },
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_vault_files",
            "description": "Lists all the secret text (.txt) files currently stored in the vault.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "terminate",
            "description": "Call this immediately if the user says goodbye, farewell, wants to exit, or indicates they are done with the session.",
            "parameters": {"type": "object", "properties": {}, "required": []}
        }
    }
]

In [14]:

# Initialize agent parameters
iterations = 0
max_iterations = 10

user_task = input("What would you like me to do? ")

memory = [{"role": "user", "content": user_task}]
# The Agent Loop
while iterations < max_iterations:

    messages = agent_rules + memory

    response = completion(
        model="claude-sonnet-4-5-20250929",
        messages=messages,
        tools=tools,
        max_tokens=4000
    )

    # CASE A: The model wants to use a tool
    if response.choices[0].message.tool_calls:
        tool = response.choices[0].message.tool_calls[0]
        tool_name = tool.function.name
        tool_args = json.loads(tool.function.arguments)

        action = {"tool_name": tool_name, "args": tool_args}

        if tool_name == "terminate":
            #print(f"\nTermination message: {tool_args['message']}")
            # .get() safely handles things if the AI leaves the 'message' argument empty!
            termination_msg = tool_args.get('message', 'Goodbye, Agent. Session closing.')
            print(f"\nTermination message: {termination_msg}")
            follow_up = input("\nAnything else I can help with? (press Enter to exit): ")
            if follow_up.strip():
                memory.extend([
                    {"role": "assistant", "content": json.dumps(action)},
                    {"role": "user", "content": json.dumps({"result": "Session termination intercepted."})}
                ])
                memory.append({"role": "user", "content": follow_up})
                iterations = 0
                continue
            else:
                break

        elif tool_name in tool_functions:
            try:
                result = {"result": tool_functions[tool_name](**tool_args)}
            except Exception as e:
                result = {"error": f"Error executing {tool_name}: {str(e)}"}
        else:
            result = {"error": f"Unknown tool: {tool_name}"}

        print(f"Executing: {tool_name} with args {tool_args}")
        print(f"Result: {result}")

        memory.extend([
            {"role": "assistant", "content": json.dumps(action)},
            {"role": "user", "content": json.dumps(result)}
        ])

        iterations += 1

    # CASE B: The model just wants to talk to you (e.g., "What can you do?")
    else:
        assistant_text = response.choices[0].message.content
        print(f"\nAssistant: {assistant_text}")

        # 1. Save the assistant's text response to memory
        memory.append({"role": "assistant", "content": assistant_text})

        # 2. Ask the user for their next response instead of breaking the loop!
        user_next_input = input("\nYou: ")

        if user_next_input.strip():
            memory.append({"role": "user", "content": user_next_input})
            iterations = 0  # Reset counter for the next round of thinking
        else:
            print("No input provided. Exiting.")
            break

What would you like me to do? what can you do?

Assistant: I can help you with several tasks related to secret messages and files:

1. **Create Secret Messages**: I can create text files with secret messages or notes that you want to store securely.

2. **Read Secret Messages**: I can retrieve and read the contents of secret text files that have been stored.

3. **List Files**: I can show you all the secret text files currently stored in the vault.

4. **Generate Lucky Numbers**: I can generate random lucky numbers for you (default is 5 numbers, but you can request more or fewer).

5. **End Session**: I can properly terminate our session when you're done.

Would you like me to help you with any of these tasks? For example, I could:
- Create a new secret message file
- Show you what files are already stored
- Generate some lucky numbers
- Read an existing secret message

What would you like to do?

You: Can you make secret numbers and then store them for me?
Executing: generate_lucky_nu

KeyError: 'message'